# HistoVision — обучение сегментационной модели на BCSS (Colab GPU)

Прогоняется через официальное расширение **Colab для VS Code** (Kernel → Select Kernel → Colab → New Colab Server → GPU/T4), либо напрямую в браузере на colab.research.google.com.

Код тянется из GitHub-репозитория, ветка `feature/segmentation` — см. `REPO_URL` в первой code-ячейке.

**Что делает этот ноутбук:**
1. Проверяет GPU-рантайм.
2. Клонирует репозиторий и ставит зависимости.
3. Скачивает и готовит датасет BCSS (`prepare_bcss.py download`).
4. Обучает DeepLabV3+/ResNet-18 (`train.py`) на GPU.
5. Сохраняет чекпоинт и скачивает его на локальную машину — для дальнейшей интеграции в `src/inference/`.

In [ ]:
# 1. Проверка GPU
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (нет GPU-рантайма — выберите GPU в Select Kernel)")

In [ ]:
# 2. Клонирование репозитория
REPO_URL = "https://github.com/soon00sad/histo_learn.git"
BRANCH = "feature/segmentation"

!git clone --branch {BRANCH} --single-branch {REPO_URL} /content/HistoVision
%cd /content/HistoVision

In [ ]:
# 3. Зависимости. torch/torchvision в Colab уже стоят (обычно свежее requirements.txt
# и с CUDA-сборкой) — их не трогаем, ставим только то, что нужно поверх.
!pip install -q segmentation-models-pytorch==0.5.0 gdown==6.1.0 opencv-python-headless==4.10.0.84 pydantic==2.7.4 PyYAML==6.0.1

In [ ]:
# 4. Подготовка BCSS. Полный датасет с Google Drive — может быть медленно/большим,
# это нормально на GPU-машине с нормальной сетью (в отличие от dev-машины автора).
# --limit можно убрать для полного датасета, либо оставить для первого прогона.
!python -m src.training.prepare_bcss download --out data/bcss --limit 60

In [ ]:
# 5. Обучение. Подберите --epochs/--batch-size под реальный размер датасета и GPU.
!python -m src.training.train \
    --data-dir data/bcss \
    --out models/segmentation.pth \
    --encoder resnet18 \
    --epochs 40 \
    --batch-size 16 \
    --crop-size 256 \
    --device cuda

In [ ]:
# 6. Скачать чекпоинт локально (для интеграции в src/inference/ на dev-машине).
# В браузерном Colab сработает files.download; из VS Code проще открыть файл
# в проводнике Colab-рантайма и скачать вручную, либо загрузить в Google Drive:
try:
    from google.colab import files
    files.download("models/segmentation.pth")
except Exception as e:
    print("files.download недоступен в этом окружении (", e, ") — скачайте models/segmentation.pth вручную.")